<a href="https://colab.research.google.com/github/arlylelu25-debug/rtm1_2501010335/blob/main/Notebook/akuisisi_openstreetmap_bali.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Akuisisi Data Tempat Wisata di Provinsi Bali Menggunakan OpenStreetMap

In [1]:
import requests
import pandas as pd
import json
import hashlib
from pathlib import Path
from datetime import datetime, timezone
from zoneinfo import ZoneInfo
from IPython.display import display

NAMA = "Maria Indriani Lelu"
NIM = "2501010335"
KAMPUS = "INSTIKI Bali"

USER_AGENT = f"RTM1-BIFP243-{NIM} ({NAMA}; tugas akademik INSTIKI)"

print("Nama:", NAMA)
print("NIM:", NIM)
print("Kampus:", KAMPUS)

Nama: Maria Indriani Lelu
NIM: 2501010335
Kampus: INSTIKI Bali


2. Menentukan Folder Penyimpanan

In [2]:
ROOT_DIR = Path.cwd()

DATA_DIR = ROOT_DIR / "data" / "raw"
DOCS_DIR = ROOT_DIR / "docs"

DATA_DIR.mkdir(parents=True, exist_ok=True)
DOCS_DIR.mkdir(parents=True, exist_ok=True)

print("Folder penyimpanan berhasil disiapkan.")

Folder penyimpanan berhasil disiapkan.


3. Menentukan Sumber Data OpenStreetMap

In [3]:
OVERPASS_URL = "https://overpass-api.de/api/interpreter"

query = """
[out:json][timeout:180];
area["name"="Bali"]["admin_level"="4"]->.searchArea;
(
  node["tourism"~"^(attraction|museum|viewpoint|theme_park|zoo|aquarium)$"](area.searchArea);
  way["tourism"~"^(attraction|museum|viewpoint|theme_park|zoo|aquarium)$"](area.searchArea);
  relation["tourism"~"^(attraction|museum|viewpoint|theme_park|zoo|aquarium)$"](area.searchArea);
);
out center;
"""

print("Query OpenStreetMap berhasil disiapkan.")

Query OpenStreetMap berhasil disiapkan.


4. Proses Akuisisi Data

In [5]:

headers = {"User-Agent": USER_AGENT}

response = requests.post(
    OVERPASS_URL,
    data={"data": query},
    headers=headers,
    timeout=180
)

response.raise_for_status()

hasil = response.json()
elements = hasil.get("elements", [])

print("Jumlah data dari OpenStreetMap:", len(elements))

Jumlah data dari OpenStreetMap: 727


5. Pengolahan Data

In [6]:
records = []

for item in elements:
    tags = item.get("tags", {})

    lat = item.get("lat")
    lon = item.get("lon")

    if lat is None or lon is None:
        center = item.get("center", {})
        lat = center.get("lat")
        lon = center.get("lon")

    if lat is None or lon is None:
        continue

    records.append({
        "nama_wisata": tags.get("name", ""),
        "kategori": tags.get("tourism", ""),
        "latitude": lat,
        "longitude": lon,
        "osm_id": item.get("id"),
        "osm_type": item.get("type", ""),
        "sumber": "OpenStreetMap"
    })

kolom = [
    "nama_wisata",
    "kategori",
    "latitude",
    "longitude",
    "osm_id",
    "osm_type",
    "sumber"
]

df = pd.DataFrame(records, columns=kolom)

df = df.drop_duplicates(
    subset=["osm_type", "osm_id"]
).reset_index(drop=True)

print("Jumlah data setelah pengolahan:", len(df))
display(df.head(10))

Jumlah data setelah pengolahan: 727


,nama_wisata,kategori,latitude,longitude,osm_id,osm_type,sumber
0,Small Beautiful Waterfall,attraction,-8.175709,115.106360,275573942,node,OpenStreetMap
1,Sunrise viewpoint of Tanah Lot Temple,viewpoint,-8.619558,115.086291,597770793,node,OpenStreetMap
2,Bali Zoo,zoo,-8.591266,115.264542,603098166,node,OpenStreetMap
3,Bali Butterfly Park,attraction,-8.479459,115.134629,641637893,node,OpenStreetMap
4,Museum Subak,museum,-8.544466,115.138400,838277764,node,OpenStreetMap
5,,viewpoint,-8.808599,115.167147,945204760,node,OpenStreetMap
6,Paintball,attraction,-8.817786,115.163878,945590562,node,OpenStreetMap
7,Pura Susunan Wadonan,attraction,-8.740111,115.237100,989867608,node,OpenStreetMap
8,Taman Burung Bali Bird Park,zoo,-8.611134,115.259644,989944122,node,OpenStreetMap
9,Museum Purbakala,museum,-8.511945,115.299746,989944131,node,OpenStreetMap


6. Validasi Data

In [7]:
if df.empty:
    raise ValueError("Data kosong. Periksa kembali koneksi API.")

print("Jumlah baris:", len(df))
print("Jumlah kolom:", len(df.columns))

print("\nPemeriksaan data kosong:")
display(df.isna().sum().to_frame("Jumlah Data Kosong"))

Jumlah baris: 727
Jumlah kolom: 7

Pemeriksaan data kosong:


,Jumlah Data Kosong
nama_wisata,0
kategori,0
latitude,0
longitude,0
osm_id,0
osm_type,0
sumber,0


7. Menyimpan Dataset ke CSV

In [8]:
tanggal = datetime.now(
    ZoneInfo("Asia/Makassar")
).strftime("%Y%m%d")

csv_path = DATA_DIR / f"openstreetmap_bali_{tanggal}.csv"

df.to_csv(
    csv_path,
    index=False,
    encoding="utf-8-sig"
)

print("Dataset berhasil disimpan:", csv_path)

Dataset berhasil disimpan: /content/data/raw/openstreetmap_bali_20261116.csv


8. Membuat Provenance Data

In [9]:
file_bytes = csv_path.read_bytes()

provenance = {
    "judul": "Akuisisi Data Tempat Wisata di Provinsi Bali Menggunakan OpenStreetMap",
    "nama": NAMA,
    "nim": NIM,
    "kampus": KAMPUS,
    "sumber": "OpenStreetMap melalui Overpass API",
    "endpoint": OVERPASS_URL,
    "waktu_wita": datetime.now(
        ZoneInfo("Asia/Makassar")
    ).isoformat(),
    "waktu_utc": datetime.now(timezone.utc).isoformat(),
    "jumlah_record": int(len(df)),
    "nama_kolom": list(df.columns),
    "nama_file": csv_path.name,
    "ukuran_file_byte": len(file_bytes),
    "md5": hashlib.md5(file_bytes).hexdigest()
}

with open(DOCS_DIR / "provenance.json", "w", encoding="utf-8") as file:
    json.dump(provenance, file, ensure_ascii=False, indent=4)

print("Provenance berhasil dibuat.")

Provenance berhasil dibuat.


9. Menampilkan Hasil Akhir

In [10]:
print("HASIL AKUISISI DATA WISATA BALI")
print("Nama:", NAMA)
print("NIM:", NIM)
print("Jumlah data:", len(df))
print("Jumlah kolom:", len(df.columns))

display(df.head(10))

print("\nAtribusi: © OpenStreetMap contributors")
print("Proses akuisisi data selesai.")

HASIL AKUISISI DATA WISATA BALI
Nama: Maria Indriani Lelu
NIM: 2501010335
Jumlah data: 727
Jumlah kolom: 7


,nama_wisata,kategori,latitude,longitude,osm_id,osm_type,sumber
0,Small Beautiful Waterfall,attraction,-8.175709,115.106360,275573942,node,OpenStreetMap
1,Sunrise viewpoint of Tanah Lot Temple,viewpoint,-8.619558,115.086291,597770793,node,OpenStreetMap
2,Bali Zoo,zoo,-8.591266,115.264542,603098166,node,OpenStreetMap
3,Bali Butterfly Park,attraction,-8.479459,115.134629,641637893,node,OpenStreetMap
4,Museum Subak,museum,-8.544466,115.138400,838277764,node,OpenStreetMap
5,,viewpoint,-8.808599,115.167147,945204760,node,OpenStreetMap
6,Paintball,attraction,-8.817786,115.163878,945590562,node,OpenStreetMap
7,Pura Susunan Wadonan,attraction,-8.740111,115.237100,989867608,node,OpenStreetMap
8,Taman Burung Bali Bird Park,zoo,-8.611134,115.259644,989944122,node,OpenStreetMap
9,Museum Purbakala,museum,-8.511945,115.299746,989944131,node,OpenStreetMap



Atribusi: © OpenStreetMap contributors
Proses akuisisi data selesai.
